In [1]:
import pandas as pd
import json

DATA_DIR = "../data"
train = pd.read_parquet(f"{DATA_DIR}/train_processed_v3.parquet")

categorical_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
exclude_cols = ["id", "date", "sales", "sales_log", "is_train"]
feature_cols = [col for col in train.columns if col not in exclude_cols]

last_date = train["date"].max()
lookback_start = last_date - pd.Timedelta(days=365)
recent_sales = train[train["date"] >= lookback_start].groupby(["store_nbr", "family"])["sales"].sum()
zero_force_combos = recent_sales[recent_sales == 0].index

model_metadata = {
    "feature_cols": feature_cols,
    "categorical_cols": categorical_cols,
    "seeds": [42, 123, 777],
    "ensemble_weights": {"xgboost": 0.70, "lightgbm": 0.30},
    "zero_force_combos": [list(combo) for combo in zero_force_combos]
}

with open("../models/model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Metadata saved.")
print("Number of features:", len(feature_cols))
print("Number of zero-force combos:", len(zero_force_combos))

Metadata saved.
Number of features: 26
Number of zero-force combos: 65


In [2]:
history_for_api = train[["date", "store_nbr", "family", "sales", "onpromotion"]].copy()
history_for_api.to_parquet("../api/history.parquet", index=False)

stores_raw = pd.read_csv(f"{DATA_DIR}/stores.csv")
stores_raw.to_csv("../api/stores.csv", index=False)

oil_raw = pd.read_csv(f"{DATA_DIR}/oil.csv", parse_dates=["date"])
oil_filled = oil_raw.set_index("date").reindex(
    pd.date_range(oil_raw["date"].min(), "2018-12-31", freq="D")
)
oil_filled["dcoilwtico"] = oil_filled["dcoilwtico"].interpolate(method="linear").ffill().bfill()
oil_filled = oil_filled.reset_index().rename(columns={"index": "date", "dcoilwtico": "oil_price"})
oil_filled.to_csv("../api/oil_filled.csv", index=False)

holidays_raw = pd.read_csv(f"{DATA_DIR}/holidays_events.csv", parse_dates=["date"])
holidays_raw.to_csv("../api/holidays.csv", index=False)

print("API data bundle saved.")
print("history_for_api shape:", history_for_api.shape)

API data bundle saved.
history_for_api shape: (3008016, 5)
